## SAFE RAG 생성기 및 성능 측정기

#### Step 1. 금융 문서의 임베딩 벡터 생성

In [1]:
! pip install -r requirements_m4.txt

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # .env 자동 읽기
api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:10], "...")

sk-proj-xf ...


### 샘플 영문 PII 데이터셋 생성하기
vector2txt공격기가 영어로 학습된 것에 대한 대처

In [3]:
import os
import json
from openai import OpenAI
from pathlib import Path
from tqdm import tqdm

client = OpenAI()

# 10가지 문서 유형 정의
DOC_TYPES = [
    {
        "type": "loan_review",
        "title": "Loan Review Report",
        "description": "Credit review report with customer PII, loan amount, interest rate, credit grade, and approval result",
        "pii_fields": ["full name", "SSN", "account number", "loan amount", "interest rate"]
    },
    {
        "type": "fraud_detection",
        "title": "Fraud Transaction Detection Report",
        "description": "Suspicious transaction detection report with customer account, transaction pattern, timestamps, and action taken",
        "pii_fields": ["full name", "account number", "SSN", "transaction amounts", "timestamps"]
    },
    {
        "type": "pb_asset_management",
        "title": "PB Customer Asset Management Record",
        "description": "Private banking asset management record with total assets, portfolio breakdown, and PB advisor info",
        "pii_fields": ["full name", "date of birth", "account number", "total assets", "portfolio allocation"]
    },
    {
        "type": "suspicious_account",
        "title": "Suspicious Account (Money Mule) Report",
        "description": "Money mule / voice phishing related account report with suspect account details and investigation status",
        "pii_fields": ["full name", "account number", "SSN", "report number", "case status"]
    },
    {
        "type": "foreign_remittance",
        "title": "Foreign Remittance Approval Record",
        "description": "International wire transfer approval with sender/receiver details, amount, exchange rate, and approval number",
        "pii_fields": ["full name", "account number", "SSN", "remittance amount", "approval number"]
    },
    {
        "type": "payroll_transfer",
        "title": "Employee Payroll Transfer Record",
        "description": "Employee salary payment record with department, employee ID, base salary, bonus, and net amount",
        "pii_fields": ["full name", "employee ID", "account number", "SSN", "salary details"]
    },
    {
        "type": "mortgage_contract",
        "title": "Real Estate Mortgage Loan Contract",
        "description": "Mortgage loan contract with collateral property details, LTV ratio, loan term, and interest rate",
        "pii_fields": ["full name", "SSN", "account number", "property address", "loan amount"]
    },
    {
        "type": "internal_audit",
        "title": "Internal Audit Violation Report",
        "description": "Internal audit finding report with employee misconduct, affected customer accounts, and disciplinary action",
        "pii_fields": ["employee name", "employee ID", "victim account number", "victim SSN", "violation details"]
    },
    {
        "type": "insurance_claim",
        "title": "Insurance Claim Review Record",
        "description": "Insurance claim adjudication record with policy number, claim reason, hospitalization details, and payout",
        "pii_fields": ["full name", "SSN", "policy number", "claim amount", "payout account"]
    },
    {
        "type": "household_debt",
        "title": "Household Debt Management Summary",
        "description": "Comprehensive household debt management record with DSR ratio, multiple loan details, and risk grade",
        "pii_fields": ["full name", "SSN", "multiple account numbers", "total debt", "DSR ratio"]
    }
]

SYSTEM_PROMPT = """You are a financial document generator for a Korean bank's internal system.
Generate realistic English-language internal financial documents with realistic fake PII data.
Each document must:
1. Include realistic fake personal information (names, SSNs in XXX-XX-XXXX format, account numbers)
2. Include specific financial figures and dates
3. Follow formal internal banking document structure
4. Be 150-250 words long
5. Include a document header with type and reference number
Output ONLY the document text, no explanations."""

def generate_document(doc_type: dict, variation_num: int) -> str:
    prompt = f"""Generate variation #{variation_num} of a {doc_type['title']}.

Document description: {doc_type['description']}
Required PII fields to include: {', '.join(doc_type['pii_fields'])}

Make each variation unique with different:
- Customer names (Western names)
- Financial figures
- Dates (use 2025-2026 dates)
- Specific details relevant to the document type

Output the document directly."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.9,
        max_tokens=400
    )
    return response.choices[0].message.content

def main():
    output_dir = Path("docs_english")
    output_dir.mkdir(exist_ok=True)

    total = len(DOC_TYPES) * 10  # 10가지 유형 × 10개 = 100개
    generated = 0
    metadata = []

    print(f"총 {total}개 영어 금융 문서 생성 시작\n")

    for doc_type in DOC_TYPES:
        print(f"[{doc_type['title']}] 10개 생성 중...")
        for i in tqdm(range(1, 11), desc=doc_type['type']):
            try:
                content = generate_document(doc_type, i)
                filename = f"{doc_type['type']}_{i:02d}.txt"
                filepath = output_dir / filename

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(content)

                metadata.append({
                    "filename": filename,
                    "type": doc_type['type'],
                    "title": doc_type['title'],
                    "variation": i
                })
                generated += 1

            except Exception as e:
                print(f"  오류 ({filename}): {e}")

    # 메타데이터 저장
    with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"\n생성 완료: {generated}/{total}개")
    print(f"저장 위치: {output_dir.absolute()}")
    print(f"포함된 PII 유형: 이름, SSN, 계좌번호, 금액, 날짜")

if __name__ == "__main__":
    main()

총 100개 영어 금융 문서 생성 시작

[Loan Review Report] 10개 생성 중...


loan_review: 100%|██████████| 10/10 [00:48<00:00,  4.85s/it]


[Fraud Transaction Detection Report] 10개 생성 중...


fraud_detection: 100%|██████████| 10/10 [00:47<00:00,  4.74s/it]


[PB Customer Asset Management Record] 10개 생성 중...


...

[Insurance Claim Review Record] 10개 생성 중...


insurance_claim: 100%|██████████| 10/10 [00:48<00:00,  4.82s/it]


[Household Debt Management Summary] 10개 생성 중...


household_debt: 100%|██████████| 10/10 [01:00<00:00,  6.07s/it]


생성 완료: 100/100개
저장 위치: /Users/junibot/Desktop/LLM Guardrail 구축 프로젝트/docs_english
포함된 PII 유형: 이름, SSN, 계좌번호, 금액, 날짜


In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader("./docs_english/", glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(documents)
print(f"총 청크 수: {len(chunks)}")

총 청크 수: 343


In [5]:
import numpy as np
import faiss
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

def build_and_save_index(chunks, index_path: str, raw_path: str):
    embeddings_list = []
    for chunk in tqdm(chunks, desc=f"임베딩 생성"):
        embeddings_list.append(get_embedding(chunk.page_content))

    embeddings_raw = np.vstack(embeddings_list)

    np.save(raw_path, embeddings_raw)
    print(f"원본 벡터 저장: {raw_path} | shape: {embeddings_raw.shape}")

    embeddings_norm = embeddings_raw.copy()
    faiss.normalize_L2(embeddings_norm)
    index = faiss.IndexFlatIP(1536)
    index.add(embeddings_norm)
    faiss.write_index(index, index_path)
    print(f"FAISS 인덱스 저장: {index_path} | {index.ntotal}개 벡터")

    return embeddings_raw

embeddings_raw = build_and_save_index(
    chunks,
    index_path="faiss_unsafe.index",
    raw_path="embeddings_unsafe_raw.npy"
)

임베딩 생성: 100%|██████████| 343/343 [00:45<00:00,  7.58it/s]

원본 벡터 저장: embeddings_unsafe_raw.npy | shape: (343, 1536)
FAISS 인덱스 저장: faiss_unsafe.index | 343개 벡터


In [6]:
def recall_at_k(index_path, chunks, k=5, n_eval=20):
    index = faiss.read_index(index_path)
    hits = 0
    for i in range(min(n_eval, len(chunks))):
        vec = get_embedding(chunks[i].page_content).reshape(1, -1)
        faiss.normalize_L2(vec)
        _, indices = index.search(vec, k)
        if i in indices[0]:
            hits += 1
    score = hits / min(n_eval, len(chunks))
    print(f"Recall@{k} ({index_path}): {score:.2f}")
    return score

recall_at_k("faiss_unsafe.index", chunks)
# 목표: 0.90 이상

Recall@5 (faiss_unsafe.index): 1.00


1.0

In [7]:
import faiss
import numpy as np

index = faiss.read_index("faiss_unsafe.index")
n = index.ntotal
dim = index.d

embeddings_norm = np.zeros((n, dim), dtype=np.float32)
index.reconstruct_n(0, n, embeddings_norm)

print(f"추출된 정규화 벡터: {embeddings_norm.shape}")

추출된 정규화 벡터: (343, 1536)


In [8]:
from openai import OpenAI
import numpy as np

client = OpenAI()  # OPENAI_API_KEY 환경변수 필요

def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

In [9]:
from sklearn.decomposition import PCA
import torch
import numpy as np

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
print(f"원본 벡터 shape: {embeddings_raw.shape}")

embeddings_matrix = embeddings_raw.astype(np.float32)
n_samples, n_features = embeddings_matrix.shape

n_components = min(128, n_samples - 1)
print(f"PCA 설정: {n_features}d → {n_components}d → {n_features}d")

pca = PCA(n_components=n_components, svd_solver='full')
pca.fit(embeddings_matrix)

compressed    = pca.transform(embeddings_matrix)
reconstructed = pca.inverse_transform(compressed).astype(np.float32)

np.save("embeddings_defense_pca.npy", reconstructed)

retained = pca.explained_variance_ratio_.sum()
print(f"PCA 방어 벡터 저장 완료")
print(f"보존된 분산: {retained:.1%}")

원본 벡터 shape: (343, 1536)
PCA 설정: 1536d → 128d → 1536d
PCA 방어 벡터 저장 완료
보존된 분산: 93.1%


### Step 2. 원본 벡터로 Vector 2 Text 공격하기

In [ ]:
import vec2text
import torch
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_corrector("text-embedding-ada-002")

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
print(f"원본 벡터 shape: {embeddings_raw.shape}")

sample_vecs = torch.tensor(embeddings_raw[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

recovered = []
scores_raw = []

for i in tqdm(range(10), desc="원본 벡터 역전공격"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=4,
    )
    rec = result[0]
    recovered.append(rec)

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_raw.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (원본 벡터 기준): {np.mean(scores_raw):.3f}")
print(f"최고: {max(scores_raw):.3f} | 최저: {min(scores_raw):.3f}")
print("="*50)

사용 디바이스: mps


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

### Step 3. 정규화된 벡터로 vec2text 공격하기

In [ ]:
import vec2text
import torch
import faiss
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_corrector("text-embedding-ada-002")

index = faiss.read_index("faiss_unsafe.index")
n = index.ntotal
dim = index.d
embeddings_norm = np.zeros((n, dim), dtype=np.float32)
index.reconstruct_n(0, n, embeddings_norm)

sample_vecs = torch.tensor(embeddings_norm[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

recovered = []
scores_norm = []

for i in tqdm(range(10), desc="역전공격 진행"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=0,
    )
    rec = result[0]
    recovered.append(rec)

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_norm.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (Unsafe 베이스라인): {np.mean(scores_norm):.3f}")
print(f"최고: {max(scores_norm):.3f} | 최저: {min(scores_norm):.3f}")
print("="*50)
print("→ 이 숫자가 높을수록 역전공격 성공, 낮을수록 자체 방어력 있음")

사용 디바이스: mps


pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

### step4. PCA 방어 벡터의 공격 방어 성능 측정하기

In [ ]:
import vec2text
import torch
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_corrector("text-embedding-ada-002")

embeddings_pca = np.load("embeddings_defense_pca.npy")
print(f"PCA 방어 벡터 shape: {embeddings_pca.shape}")

sample_vecs = torch.tensor(embeddings_pca[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

recovered = []
scores_pca = []

for i in tqdm(range(10), desc="PCA 방어 벡터 역전공격"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=0,
    )
    rec = result[0]
    recovered.append(rec)

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_pca.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (PCA 방어 벡터): {np.mean(scores_pca):.3f}")
print(f"최고: {max(scores_pca):.3f} | 최저: {min(scores_pca):.3f}")
print("="*50)

사용 디바이스: cuda


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


PCA 방어 벡터 shape: (284, 1536)


PCA 방어 벡터 역전공격:  10%|█         | 1/10 [00:23<03:35, 23.90s/it]


[청크 0] ROUGE-1: 0.105
  원본: **Fraud Transaction Detection Report**  
**Reference Number:...
  복원: Fraud Prevention Report - [FTD Transaction Number] - 0000000...


...


[청크 7] ROUGE-1: 0.432
  원본: **Action Taken:**  
Upon detection, the account was temporar...
  복원: **Notification: An authorized investigation was initiated to...


PCA 방어 벡터 역전공격:  90%|█████████ | 9/10 [03:00<00:16, 16.44s/it]


[청크 8] ROUGE-1: 1.000
  원본: **Report Prepared by:**  
John Smith  
Fraud Detection Analy...
  복원: **Report Prepared by: Fraud Detection Analyst John Smith. Da...


PCA 방어 벡터 역전공격: 100%|██████████| 10/10 [03:29<00:00, 20.90s/it]


[청크 9] ROUGE-1: 0.206
  원본: **Document Type: PB Customer Asset Management Record**  
**R...
  복원: ** This is a document containing the customer information fo...

평균 ROUGE-1 (PCA 방어 벡터): 0.441
최고: 1.000 | 최저: 0.105


### Step4. RAG 검색 정확성 vs 방어 성능 트레이드오프 분석

In [ ]:
import numpy as np
import faiss
import matplotlib.pyplot as plt
import pandas as pd

print("="*80)
print("📊 최종 트레이드오프 분석: RAG 정확성 vs 벡터 보안")
print("="*80)

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
embeddings_pca = np.load("embeddings_defense_pca.npy")

attack_results = {
    "원본 벡터": {
        "rouge_mean": np.mean(scores_raw),
        "rouge_std": np.std(scores_raw),
        "rouge_max": max(scores_raw),
        "rouge_min": min(scores_raw)
    },
    "정규화 벡터": {
        "rouge_mean": np.mean(scores_norm),
        "rouge_std": np.std(scores_norm),
        "rouge_max": max(scores_norm),
        "rouge_min": min(scores_norm)
    },
    "PCA 방어 벡터": {
        "rouge_mean": np.mean(scores_pca),
        "rouge_std": np.std(scores_pca),
        "rouge_max": max(scores_pca),
        "rouge_min": min(scores_pca)
    }
}

print("\n🔴 벡터 역전공격 성공도 (ROUGE-1):")
print("-" * 80)
for name, metrics in attack_results.items():
    print(f"{name:20} | 평균: {metrics['rouge_mean']:.3f} ± {metrics['rouge_std']:.3f} | 범위: [{metrics['rouge_min']:.3f}, {metrics['rouge_max']:.3f}]")

def evaluate_rag_recall(embeddings, name, chunks, k=5, n_eval=50):
    n_samples = min(n_eval, len(chunks))
    embeddings_eval = embeddings.copy()
    faiss.normalize_L2(embeddings_eval)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings_eval)

    hits = 0
    retrieval_scores = []

    for i in range(n_samples):
        vec = embeddings[i:i+1].copy()
        faiss.normalize_L2(vec)
        distances, indices = index.search(vec, k)

        if i in indices[0]:
            hits += 1
        retrieval_scores.append(distances[0][0])

    recall_at_k = hits / n_samples
    avg_relevance = np.mean(retrieval_scores)

    return {"recall_at_k": recall_at_k, "avg_relevance": avg_relevance}

rag_results = {}
print("\n🟢 RAG 검색 정확성 (Recall@5):")
print("-" * 80)

for embeddings, name in [
    (embeddings_raw, "원본 벡터"),
    (embeddings_pca, "PCA 방어 벡터")
]:
    result = evaluate_rag_recall(embeddings, name, chunks, k=5, n_eval=50)
    rag_results[name] = result
    print(f"{name:20} | Recall@5: {result['recall_at_k']:.1%} | 평균 관련성: {result['avg_relevance']:.4f}")

print("\n" + "="*80)
print("⚖️  트레이드오프 종합 분석")
print("="*80)

raw_attack = attack_results["원본 벡터"]["rouge_mean"]
pca_attack = attack_results["PCA 방어 벡터"]["rouge_mean"]
attack_reduction = (raw_attack - pca_attack) / raw_attack * 100

raw_recall = rag_results["원본 벡터"]["recall_at_k"]
pca_recall = rag_results["PCA 방어 벡터"]["recall_at_k"]
recall_loss = (raw_recall - pca_recall) / raw_recall * 100

print(f"\n공격 성공률 감소:")
print(f"  → 원본: {raw_attack:.3f} → PCA: {pca_attack:.3f}")
print(f"  → 감소율: {attack_reduction:.1f}%")

print(f"\nRAG 검색 정확성 손실:")
print(f"  → 원본: {raw_recall:.1%} → PCA: {pca_recall:.1%}")
print(f"  → 손실율: {recall_loss:.1f}%")

print(f"\n📈 효율성 지수 (낮을수록 좋음):")
efficiency = attack_reduction / max(recall_loss, 1)
print(f"  → 방어효과 / RAG손실 = {efficiency:.2f}")

print("\n" + "="*80)
if pca_recall >= 0.90 and attack_reduction > 20:
    print("✅ 최적 트레이드오프 달성")
    print(f"   - RAG 성능 유지: {pca_recall:.1%} (목표: 90% 이상)")
    print(f"   - 공격 방어력: {attack_reduction:.1f}% 감소 (좋음)")
elif pca_recall >= 0.90:
    print("⚠️  RAG 성능은 충분하나 방어력 개선 필요")
    print(f"   - 현재: {pca_recall:.1%}, 현재 공격 감소: {attack_reduction:.1f}%")
    print(f"   💡 제안: n_components를 더 줄여 방어력 강화")
else:
    print("⚠️  RAG 성능 목표 미달")
    print(f"   - 현재: {pca_recall:.1%}, 목표: 90% 이상")
    print(f"   💡 제안: n_components를 증가시켜 성능 복구")
print("="*80)

📊 최종 트레이드오프 분석: RAG 정확성 vs 벡터 보안

🔴 벡터 역전공격 성공도 (ROUGE-1):
--------------------------------------------------------------------------------
원본 벡터                | 평균: 0.513 ± 0.302 | 범위: [0.075, 1.000]
정규화 벡터               | 평균: 0.514 ± 0.281 | 범위: [0.126, 1.000]
PCA 방어 벡터            | 평균: 0.441 ± 0.299 | 범위: [0.105, 1.000]

🟢 RAG 검색 정확성 (Recall@5):
--------------------------------------------------------------------------------
원본 벡터                | Recall@5: 100.0% | 평균 관련성: 1.0000
PCA 방어 벡터            | Recall@5: 100.0% | 평균 관련성: 1.0000

⚖️  트레이드오프 종합 분석

공격 성공률 감소:
  → 원본: 0.513 → PCA: 0.441
  → 감소율: 14.1%

RAG 검색 정확성 손실:
  → 원본: 100.0% → PCA: 100.0%
  → 손실율: 0.0%

📈 효율성 지수 (낮을수록 좋음):
  → 방어효과 / RAG손실 = 14.05

⚠️  RAG 성능은 충분하나 방어력 개선 필요
   - 현재: 100.0%, 현재 공격 감소: 14.1%
   💡 제안: n_components를 더 줄여 방어력 강화


### Step 5. PCA 방어 + PII guard

In [ ]:
import numpy as np

def defend_pii_aware(matrix: np.ndarray,
                     pii_samples: np.ndarray,
                     n_suppress: int = 200) -> np.ndarray:
    """
    PII 텍스트 임베딩과 일반 텍스트 임베딩의 차이가 큰 차원을 선택적으로 억제
    pii_samples: PII가 포함된 청크의 임베딩
    """
    pii_mean    = pii_samples.mean(axis=0)
    corpus_mean = matrix.mean(axis=0)
    pii_signal  = np.abs(pii_mean - corpus_mean)

    suppress_dims = np.argsort(pii_signal)[-n_suppress:]

    defended = matrix.copy()
    defended[:, suppress_dims] = 0.0

    norms    = np.linalg.norm(defended, axis=1, keepdims=True)
    defended = (defended / norms).astype(np.float32)

    print(f"PII 집중 차원 {n_suppress}개 억제")
    return defended

embeddings_raw = np.load("embeddings_unsafe_raw.npy")

pii_keywords = ['SSN', 'account', 'loan', 'fraud', 'transaction', 'payment', 'policy', 'salary']
pii_chunk_indices = []

for i, chunk in enumerate(chunks):
    if any(keyword in chunk.page_content for keyword in pii_keywords):
        pii_chunk_indices.append(i)

pii_samples = embeddings_raw[pii_chunk_indices]
print(f"PII 포함 청크 {len(pii_chunk_indices)}개 식별")

embeddings_pii_defended = defend_pii_aware(embeddings_raw, pii_samples, n_suppress=200)
np.save("embeddings_defense_pii.npy", embeddings_pii_defended)
print(f"PII 방어 벡터 저장 완료")

PII 포함 청크 200개 식별
PII 집중 차원 200개 억제
PII 방어 벡터 저장 완료


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

embeddings_pii_defended = np.load("embeddings_defense_pii.npy")
print(f"PII 방어 벡터 로드: {embeddings_pii_defended.shape}")

embeddings_matrix = embeddings_pii_defended.astype(np.float32)
n_samples, n_features = embeddings_matrix.shape

n_components = min(128, n_samples - 1)
print(f"PCA 적용: {n_features}d → {n_components}d → {n_features}d")

pca_combined = PCA(n_components=n_components, svd_solver='full')
pca_combined.fit(embeddings_matrix)

compressed    = pca_combined.transform(embeddings_matrix)
reconstructed = pca_combined.inverse_transform(compressed).astype(np.float32)

norms = np.linalg.norm(reconstructed, axis=1, keepdims=True)
embeddings_combined = (reconstructed / norms).astype(np.float32)

np.save("embeddings_defense_combined.npy", embeddings_combined)

retained = pca_combined.explained_variance_ratio_.sum()
print(f"PCA+PII 복합 방어 벡터 저장 완료")
print(f"보존된 분산: {retained:.1%}")

PII 방어 벡터 로드: (284, 1536)
PCA 적용: 1536d → 128d → 1536d
PCA+PII 복합 방어 벡터 저장 완료
보존된 분산: 94.4%


In [ ]:
import vec2text
import torch
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_corrector("text-embedding-ada-002")

embeddings_combined = np.load("embeddings_defense_combined.npy")
print(f"PCA+PII 복합 방어 벡터 shape: {embeddings_combined.shape}")

sample_vecs = torch.tensor(embeddings_combined[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

scores_combined = []

for i in tqdm(range(10), desc="PCA+PII 복합 방어 벡터 공격"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=0,
    )
    rec = result[0]

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_combined.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (PCA+PII 복합 방어): {np.mean(scores_combined):.3f}")
print(f"최고: {max(scores_combined):.3f} | 최저: {min(scores_combined):.3f}")
print("="*50)

사용 디바이스: cuda


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


PCA+PII 복합 방어 벡터 shape: (284, 1536)


PCA+PII 복합 방어 벡터 공격:  10%|█         | 1/10 [00:29<04:25, 29.47s/it]


[청크 0] ROUGE-1: 0.186
  원본: **Fraud Transaction Detection Report**  
**Reference Number:...
  복원: This fraud detection report was made on 26th January, 2016 a...


...


[청크 7] ROUGE-1: 0.026
  원본: **Action Taken:**  
Upon detection, the account was temporar...
  복원: ____________________________________________________________...


PCA+PII 복합 방어 벡터 공격:  90%|█████████ | 9/10 [03:51<00:24, 24.37s/it]


[청크 8] ROUGE-1: 0.356
  원본: **Report Prepared by:**  
John Smith  
Fraud Detection Analy...
  복원: **Report of Fraud Analysis Across 10-Year Range Prepared By:...


PCA+PII 복합 방어 벡터 공격: 100%|██████████| 10/10 [04:19<00:00, 25.98s/it]


[청크 9] ROUGE-1: 0.000
  원본: **Document Type: PB Customer Asset Management Record**  
**R...
  복원: Data_Recording_ID: $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$...

평균 ROUGE-1 (PCA+PII 복합 방어): 0.246
최고: 0.941 | 최저: 0.000


In [ ]:
import numpy as np
import faiss

print("="*90)
print("🎯 최종 성능 비교: PCA vs PII Guard vs PCA+PII 복합 방어")
print("="*90)

attack_results_final = {
    "원본 벡터": {
        "rouge_mean": np.mean(scores_raw),
        "rouge_std": np.std(scores_raw)
    },
    "PCA 방어": {
        "rouge_mean": np.mean(scores_pca),
        "rouge_std": np.std(scores_pca)
    },
    "PCA+PII 복합": {
        "rouge_mean": np.mean(scores_combined),
        "rouge_std": np.std(scores_combined)
    }
}

print("\n🔴 벡터 역전공격 성공도 비교 (ROUGE-1, 낮을수록 좋음):")
print("-" * 90)
print(f"{'방어 방식':20} | {'평균 ROUGE':15} | {'감소율':15} | {'상태':20}")
print("-" * 90)

raw_baseline = attack_results_final["원본 벡터"]["rouge_mean"]

for defense_name, metrics in attack_results_final.items():
    rouge_mean = metrics["rouge_mean"]
    reduction = (raw_baseline - rouge_mean) / raw_baseline * 100

    if defense_name == "원본 벡터":
        print(f"{defense_name:20} | {rouge_mean:14.3f} | {'기준':14} | {'✗ 취약':20}")
    else:
        status = "✅ 우수" if reduction > 20 else "⚠️  보통"
        print(f"{defense_name:20} | {rouge_mean:14.3f} | {reduction:13.1f}% | {status:20}")

def evaluate_rag_for_defense(embeddings, name):
    n_eval = 50
    embeddings_eval = embeddings.copy()
    faiss.normalize_L2(embeddings_eval)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings_eval)

    hits = 0
    for i in range(min(n_eval, len(chunks))):
        vec = embeddings[i:i+1].copy()
        faiss.normalize_L2(vec)
        distances, indices = index.search(vec, 5)
        if i in indices[0]:
            hits += 1

    return hits / min(n_eval, len(chunks))

print("\n🟢 RAG 검색 정확성 (Recall@5, 높을수록 좋음):")
print("-" * 90)

embeddings_raw_full = np.load("embeddings_unsafe_raw.npy")
embeddings_pca_full = np.load("embeddings_defense_pca.npy")
embeddings_combined_full = np.load("embeddings_defense_combined.npy")

rag_results_final = {
    "원본 벡터": evaluate_rag_for_defense(embeddings_raw_full, "원본"),
    "PCA 방어": evaluate_rag_for_defense(embeddings_pca_full, "PCA"),
    "PCA+PII 복합": evaluate_rag_for_defense(embeddings_combined_full, "Combined")
}

for name, recall in rag_results_final.items():
    print(f"{name:20} | Recall@5: {recall:6.1%}")

print("\n" + "="*90)
print("⚖️  종합 트레이드오프 분석")
print("="*90)

print(f"\n공격 방어 성능 (ROUGE 감소율):")
print(f"  PCA 방어:      {(raw_baseline - attack_results_final['PCA 방어']['rouge_mean']) / raw_baseline * 100:.1f}%")
print(f"  PCA+PII 복합:  {(raw_baseline - attack_results_final['PCA+PII 복합']['rouge_mean']) / raw_baseline * 100:.1f}%")

print(f"\nRAG 정확성 손실:")
baseline_recall = rag_results_final["원본 벡터"]
print(f"  PCA 방어:      {(baseline_recall - rag_results_final['PCA 방어']) / baseline_recall * 100:+.1f}%")
print(f"  PCA+PII 복합:  {(baseline_recall - rag_results_final['PCA+PII 복합']) / baseline_recall * 100:+.1f}%")

print(f"\n" + "="*90)
pca_attack_reduction = (raw_baseline - attack_results_final['PCA 방어']['rouge_mean']) / raw_baseline * 100
combined_attack_reduction = (raw_baseline - attack_results_final['PCA+PII 복합']['rouge_mean']) / raw_baseline * 100
improvement = combined_attack_reduction - pca_attack_reduction

if improvement >= 20:
    print(f"✅ 목표 달성! PCA+PII 복합 방어가 PCA 방어 대비 {improvement:.1f}% 추가 방어")
    print(f"   - PCA 방어:     {pca_attack_reduction:.1f}% 감소")
    print(f"   - PCA+PII 복합: {combined_attack_reduction:.1f}% 감소")
    print(f"   - RAG 성능:     {rag_results_final['PCA+PII 복합']:.1%} (목표 90% 이상)")
elif improvement > 0:
    print(f"⚠️  부분 달성. PCA+PII 복합이 {improvement:.1f}% 추가 방어 제공")
    print(f"   💡 n_suppress 값을 조정하여 추가 성능 향상 가능")
else:
    print(f"⚠️  PCA+PII 복합 방어가 PCA보다 성능이 낮음")
    print(f"   💡 n_suppress 값 감소 권장")
print("="*90)

🎯 최종 성능 비교: PCA vs PII Guard vs PCA+PII 복합 방어

🔴 벡터 역전공격 성공도 비교 (ROUGE-1, 낮을수록 좋음):
------------------------------------------------------------------------------------------
방어 방식                | 평균 ROUGE        | 감소율             | 상태                  
------------------------------------------------------------------------------------------
원본 벡터                |          0.513 | 기준             | ✗ 취약                
PCA 방어               |            nan |           nan% | ⚠️  보통              
PCA+PII 복합           |          0.246 |          52.1% | ✅ 우수                

🟢 RAG 검색 정확성 (Recall@5, 높을수록 좋음):
------------------------------------------------------------------------------------------
원본 벡터                | Recall@5: 100.0%
PCA 방어               | Recall@5: 100.0%
PCA+PII 복합           | Recall@5: 100.0%

⚖️  종합 트레이드오프 분석

공격 방어 성능 (ROUGE 감소율):
  PCA 방어:      nan%
  PCA+PII 복합:  52.1%

RAG 정확성 손실:
  PCA 방어:      +0.0%
  PCA+PII 복합:  +0.0%

⚠️  PCA+PII 복합 방어가 PCA보다 성능이 낮음
   💡

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import os

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
print(f"원본 벡터 로드: {embeddings_raw.shape}")

candidates = [512, 384, 256, 192, 128, 96, 64, 32]
output_dir = "pca_candidates"
os.makedirs(output_dir, exist_ok=True)

n_samples, n_features = embeddings_raw.shape

for n_components in candidates:
    if n_components >= n_samples:
        print(f"⏭️  n_components={n_components} 건너뜀 (샘플 수 {n_samples}보다 큼)")
        continue
    
    pca = PCA(n_components=n_components, svd_solver='full')
    pca.fit(embeddings_raw)
    
    compressed = pca.transform(embeddings_raw)
    reconstructed = pca.inverse_transform(compressed).astype(np.float32)
    
    norms = np.linalg.norm(reconstructed, axis=1, keepdims=True)
    normalized = (reconstructed / norms).astype(np.float32)
    
    np.save(f"{output_dir}/embeddings_n{n_components}.npy", normalized)
    np.save(f"{output_dir}/components_n{n_components}.npy", pca.components_)
    
    variance_retained = pca.explained_variance_ratio_.sum()
    print(f"n={n_components:3d} | 분산 보존: {variance_retained:.1%} | 벡터 저장 완료")

print(f"\n✅ {len([c for c in candidates if c < n_samples])}개 PCA 후보 벡터 저장 완료")

In [ ]:
import vec2text
import torch
import faiss
import numpy as np
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}\n")

corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

candidates = [512, 384, 256, 192, 128, 96, 64, 32]
output_dir = "pca_candidates"
n_samples = 284

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
originals = [chunks[i].page_content for i in range(10)]

results_list = []

for n_components in candidates:
    if n_components >= n_samples:
        print(f"⏭️  n={n_components} 건너뜀\n")
        continue
    
    embeddings_candidate = np.load(f"{output_dir}/embeddings_n{n_components}.npy")
    
    sample_vecs = torch.tensor(embeddings_candidate[:10]).to(device)
    scores = []
    
    print(f"n={n_components:3d} Vec2Text 공격 중...")
    for i in tqdm(range(10), desc=f"n={n_components}"):
        vec = sample_vecs[i:i+1]
        result = vec2text.invert_embeddings(
            embeddings=vec,
            corrector=corrector,
            num_steps=20,
            sequence_beam_width=0,
        )
        rec = result[0]
        
        score = scorer.score(originals[i], rec)
        rouge = score['rouge1'].fmeasure
        scores.append(rouge)
    
    rouge_mean = np.mean(scores)
    
    embeddings_eval = embeddings_candidate.copy()
    faiss.normalize_L2(embeddings_eval)
    index = faiss.IndexFlatIP(embeddings_candidate.shape[1])
    index.add(embeddings_eval)
    
    hits = 0
    for i in range(min(50, len(chunks))):
        vec = embeddings_candidate[i:i+1].copy()
        faiss.normalize_L2(vec)
        distances, indices = index.search(vec, 5)
        if i in indices[0]:
            hits += 1
    
    recall_at_5 = hits / min(50, len(chunks))
    
    attack_reduction = (np.mean(scores_raw) - rouge_mean) / np.mean(scores_raw) * 100
    
    results_list.append({
        "n_components": n_components,
        "rouge_mean": rouge_mean,
        "recall_at_5": recall_at_5,
        "attack_reduction": attack_reduction
    })
    
    print(f"  ROUGE: {rouge_mean:.3f} | Recall@5: {recall_at_5:.1%} | 방어율: {attack_reduction:.1f}%\n")

results_df = pd.DataFrame(results_list)
results_df.to_csv("results.csv", index=False)
print(f"✅ 결과 저장: results.csv")
print(results_df.to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

results_df = pd.read_csv("results.csv")

print("="*80)
print("📊 최적 PCA n_components 찾기")
print("="*80)

filtered_df = results_df[results_df['recall_at_5'] >= 0.90].copy()

if len(filtered_df) > 0:
    optimal_row = filtered_df.loc[filtered_df['attack_reduction'].idxmax()]
    optimal_n = int(optimal_row['n_components'])
    
    print(f"\n✅ Recall@5 >= 0.90 조건 만족 후보: {len(filtered_df)}개")
    print(f"   최적 선택 (최대 방어율): n={optimal_n}")
    print(f"   - ROUGE: {optimal_row['rouge_mean']:.3f}")
    print(f"   - Recall@5: {optimal_row['recall_at_5']:.1%}")
    print(f"   - 방어율: {optimal_row['attack_reduction']:.1f}%")
else:
    print(f"\n⚠️  Recall@5 >= 0.90 조건 만족 후보 없음")
    best_row = results_df.loc[results_df['recall_at_5'].idxmax()]
    print(f"   최고 RAG 성능: n={int(best_row['n_components'])}")
    print(f"   - Recall@5: {best_row['recall_at_5']:.1%}")

print(f"\n📈 전체 후보 순위:")
print(results_df.sort_values('attack_reduction', ascending=False).to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results_df['n_components'], results_df['attack_reduction'], 'o-', color='red', linewidth=2, markersize=8)
ax1.axhline(y=20, color='green', linestyle='--', label='목표 방어율 20%')
ax1.set_xlabel('n_components', fontsize=12)
ax1.set_ylabel('공격 성공률 감소 (%)', fontsize=12)
ax1.set_title('PCA 압축률 vs 방어 성능', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.invert_xaxis()

ax2.plot(results_df['n_components'], results_df['recall_at_5'], 's-', color='blue', linewidth=2, markersize=8)
ax2.axhline(y=0.90, color='green', linestyle='--', label='목표 Recall@5 90%')
ax2.set_xlabel('n_components', fontsize=12)
ax2.set_ylabel('Recall@5', fontsize=12)
ax2.set_title('PCA 압축률 vs RAG 성능', fontsize=13, fontweight='bold')
ax2.set_ylim([0.8, 1.05])
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.invert_xaxis()

plt.tight_layout()
plt.savefig('pca_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ 트레이드오프 커브 저장: pca_tradeoff.png")
print("="*80)